
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 강의 - Databricks에서 에이전트 도구로 유니티 카탈로그 함수 활용

## 개요

"미션 지역의 평균 주택 가격은 얼마인가요?"라거나 "당사 우수 고객들의 고객 생애 가치(CLV)를 계산해 줘" 같은 질문을 던졌을 때, 올바른 데이터 작업과 비즈니스 로직을 자동으로 찾아 실행하여 즉각 답을 내려주는 AI 에이전트를 상상해 보십시오. 이것이 바로 에이전트 도구로 활용되는 유니티 카탈로그 함수(Unity Catalog Functions as Agent Tools)의 강력한 힘입니다.

AI 에이전트와 도구의 기본 개념을 바탕으로, 본 세션에서는 가장 실용적인 구현 방식 중 하나를 집중적으로 다룹니다. 바로 유니티 카탈로그의 SQL 및 Python 함수를 지능적이고 탐색 가능한 도구로 활용하여, AI 에이전트가 자연어 쿼리를 기반으로 이를 자동 선택하고 실행하도록 만드는 방법입니다.

본 강의에서는 이어지는 실습 데모를 위한 기술적 기반과 모범 사례를 정립합니다. 실습에서는 [AI Playground](https://docs.databricks.com/aws/en/generative-ai/agent-framework/ai-playground-agent)를 사용해 SQL 및 Python 유니티 카탈로그 함수를 에이전트 도구로 직접 구축하고 테스트해 보게 됩니다.

### 학습 목표

_이 강의가 끝나면 다음을 할 수 있을 것입니다:_

- 유니티 카탈로그(UC) 함수와 에이전트 도구 간의 근본적인 차이점 이해
- SQL 에이전트 도구와 Python 에이전트 도구의 차이점 설명
- SQL 함수 등록 방법 설명
- Python 함수를 등록하는 다양한 방법 설명
- Databricks UI를 사용하여 등록된 함수를 탐색하는 방법 이해
- AI Playground가 UC 도구와 어떻게 통합되는지 이해

## A. Unity Catalog 함수를 에이전트 도구로 이해하기

### A1. 에이전트 도구로서의 유니티 카탈로그(Unity Catalog) 함수란 무엇인가요?

시작하기에 앞서, 유니티 카탈로그(UC) 함수가 무엇인지 명확히 이해하는 것이 중요합니다. 유니티 카탈로그 도구는 내부적으로 결국 유니티 카탈로그의 사용자 정의 함수(UDF)에 불과합니다. 즉, 유니티 카탈로그 도구를 정의한다는 것은 유니티 카탈로그 내에 함수를 등록하는 것을 의미합니다. 유니티 카탈로그 UDF에 대해 자세히 알아보려면 [이 문서](https://docs.databricks.com/aws/en/udf/unity-catalog)를 참조하시기 바랍니다.

> **UDF가 무엇입니까?** 
> 유니티 카탈로그의 사용자 정의 함수(UDF)는 데이터브릭스(Databricks) 내에서 SQL과 파이썬(Python)의 기능을 확장해 줍니다. 이를 통해 고유한 맞춤형 함수를 정의하고 사용할 수 있으며, 다양한 컴퓨팅 환경 전반에서 안전하게 공유하고 관리할 수 있습니다.

**에이전트 도구로서의 유니티 카탈로그 함수** 는 AI 에이전트가 데이터 작업 및 비즈니스 로직을 수행하기 위해 동적으로 검색, 선택, 실행할 수 있도록 SQL이나 파이썬으로 작성된 유니티 카탈로그 함수를 말합니다. 호출 시 수동 프로그래밍이 필요한 기존 함수와 달리, 에이전트 도구로서의 유니티 카탈로그 함수는 다음과 같은 특징을 갖도록 설계되었습니다.

- **자체 설명 기능:** 포괄적인 메타데이터와 문서를 통해 스스로의 기능을 설명합니다.
- **맥락 적합성:** 특정 비즈니스나 분석 작업의 맥락에 맞게 적절히 적용됩니다.
- **거버넌스 제공:** 유니티 카탈로그의 보안 및 액세스 제어 메커니즘을 통해 안전하게 관리됩니다.

### A2. SQL vs Python 에이전트 도구: 주요 차이점과 사용 사례



![sql-fun-vs-agent-tool.png](../Includes/images/sql-fun-vs-agent-tool.png "sql-fun-vs-agent-tool.png")

<p>
<em>
UC SQL 함수가 에이전트 도구로 어떻게 구조화되어야 하는지에 대한 예시입니다. 
</em>
</p>

SQL 기능과 Python 기능을 언제 사용할지 이해하는 것은 효과적인 에이전트 도구 구현에 매우 중요합니다:

**SQL 에이전트 도구**
- 데이터 쿼리 및 분석 작업에 최적화됨
- `CREATE OR REPLACE FUNCTION` 문장을 사용하여 실행
- SQL 문법과 내장 함수로 제한됨
- 서버리스 모드에서만 실행
- 자동 쿼리 최적화 및 캐싱 기능 제공
- 적합한 용도: 데이터 조회, 집계(Aggregation), 필터링, 분석적 계산

**Python 에이전트 도구**
- 사용자 정의 파이썬 로직 및 복잡한 연산 실행
- 외부 API 및 라이브러리와의 연동 지원
- 유연한 실행 방식(서버리스 및 로컬 모드) 제공
- 명시적인 타입 힌트(Type Hints) 및 구글 스타일 독스트링(Docstrings) 필수 요구
- 고급 에러 처리 및 디버깅 기능 지원
- 적합한 용도: 비즈니스 로직, 외부 연동, 복잡한 알고리즘, 데이터 변환

**도구 결합**: 가장 강력한 에이전트 아키텍처는 SQL과 파이썬 도구를 함께 사용하는 것입니다. 이 구조에서는 SQL이 데이터 접근과 분석을 담당하고, 파이썬 함수가 비즈니스 로직과 외부 연동을 관리합니다.

### A3. 에이전트 도구(Agent Tools) vs 기존 함수(Traditional Functions)

에이전트 도구와 기존 함수의 차이점을 이해하는 것은 효과적인 구현을 위해 매우 중요합니다.

- **기존 함수 (Traditional Functions)**
    - 개발자가 프로그램에서 직접 호출하여 사용하도록 설계되었습니다.
    - 문서화(주석 등)에 대한 요구사항이 제한적이거나 최소한입니다.
    - 명확히 알려진 매개변수(Parameters)를 통해 명시적으로 호출됩니다.
    - 계산 효율성과 성능에 초점을 맞춥니다.

- **에이전트 도구로서의 유니티 카탈로그 함수 (Unity Catalog Functions as Agent Tools)**
    - AI 에이전트가 동적으로 탐색하고 사용할 수 있도록 설계되었습니다.
    - 풍부한 메타데이터와 포괄적인 문서화가 필수적입니다.
    - 자연어 쿼리를 바탕으로 매개변수와 사용법을 추론합니다.
    - 명확성, 해석 가능성, 그리고 에이전트의 사용 편의성에 초점을 맞춥니다.
    - 비즈니스 컨텍스트 및 구체적인 사용 예시를 포함합니다.

![python-function-diagram.png](../Includes/images/python-function-diagram.png "python-function-diagram.png")

<p align="center"><em>SQL 도구가 사용되는 방식과 유사하게, 에이전트는 자체적인 판단 능력을 활용해 UC 파이썬 도구의 실행을 추론하고 계획합니다.</em></p>

## B. 등록 방법

### B2. 함수 등록 방법

Unity Catalog는 SQL 및 Python 함수를 에이전트 도구(agent tools)로 등록하기 위한 다양한 접근 방식을 제공합니다. UC(Unity Catalog)에 등록된 함수는 UC 권한 제어를 받으므로, 세션 범위(session-scoped)나 노트북 UDF와는 등록 방식에서 차이가 있습니다.


![sql-registration-diagram](../Includes/images/sql-registration-diagram.png "sql-registration-diagram")


#### SQL 함수 등록
- [`CREATE OR REPLACE FUNCTION` 구문](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function) 사용:
  - 즉각적인 등록 및 사용 가능: 등록 즉시 활용할 수 있습니다.
  - 함수 정의 및 메타데이터 완전 제어: 함수의 정의와 메타데이터를 완벽하게 제어할 수 있습니다.
  - 기존 SQL 개발 워크플로우와의 통합: 기존에 사용하던 SQL 개발 프로세스에 그대로 통합됩니다.
  - 복잡한 SQL 로직 및 비즈니스 규칙 지원: 복잡한 SQL 로직과 비즈니스 규칙을 그대로 지원합니다.
  - 사용자 정의 환경 및 종속성 미지원: 커스텀 환경 설정이나 외부 종속성(dependencies)은 지원하지 않습니다.

![python-registration-diagram](../Includes/images/python-registration-diagram.png "python-registration-diagram")

#### Python 함수 등록
- [`DatabricksFunctionClient()`](https://docs.unitycatalog.io/ai/client/#databricks-function-client) 사용:
  - `create_python_function()` API를 통한 Python callable 직접 수용: Python 함수 객체를 직접 입력받을 수 있습니다.
  - 타입 힌트 및 독스트링(docstring) 메타데이터 자동 추출: 코드 내의 타입 힌트와 독스트링 메타데이터를 자동으로 추출합니다.
  - Unity Catalog의 3단계 네임스페이스 통합: Unity Catalog의 3단계 네임스페이스 구조와 유기적으로 통합됩니다.
  - 함수 버전 관리 및 교체 지원: 함수의 버전 관리와 기존 함수 교체 기능을 지원합니다.
  - 서버리스(운영) 및 로컬(개발) 모드 지원: 두 모드를 모두 지원하지만, 로컬 모드에서는 [SQL 기반 함수를 지원하지 않습니다](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool).
- [`CREATE OR REPLACE FUNCTION` 구문](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function) 사용: 
  - SQL 구문을 활용한 등록: SQL 도구 생성과 유사하게, Python 로직을 사용하면서도 등록은 SQL 구문을 거치는 방식으로 Python 함수를 생성할 수 있습니다.([예시](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function#create-python-functions) 참조).
  - `ENVIRONMENT`절을 통한 사용자 정의 종속성 정의: ENVIRONMENT 절을 사용하여 필요한 외부 종속성을 직접 정의할 수 있습니다.([자세한 내용](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function#define-custom-dependencies-in-python-functions)).

### B3. (선택 사항) 실행 환경 고려사항

UC Python 함수에 대한 기술적 고려사항(서버리스 vs 로컬 모드)에 대해 더 알고 싶으시다면 [이 문서](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool#running-functions-using-serverless-or-local-mode)를 참조하십시오.

## C. UI 이용한 도구 등록 검증 

함수를 유니티 카탈로그(Unity Catalog)에 등록한 후에는, LLM이 컨텍스트 및 쿼리 필터링에 사용할 메타데이터 정보를 검증할 수 있습니다. 다음은 유니티 카탈로그 내에서 도구 위치를 확인할 때 참고할 수 있는 두 가지 예시입니다.

![sql-func-validation.png](../Includes/images/sql-func-validation.png "sql-func-validation.png")

<p align="center"><em>맥락과 사용을 위해 LLM 친화적인 설명에 등록된 SQL UC 함수의 예시입니다.</em></p>

![python-tool-ui.png](../Includes/images/python-tool-ui.png "python-tool-ui.png")

<p align="center"><em> SQL 함수와 유사하게 등록된 Python 함수를 사용하면 설명, 정의 및 기타 메타데이터를 볼 수 있습니다. </em></p>

## D. 프로토타이핑을 위한 AI Playground와의 통합

### D1. AI Playground 통합

AI 플레이그라운드는 SQL 및 파이썬 유니티 카탈로그(Unity Catalog) 함수를 에이전트 도구로 테스트하고 프로토타입을 제작할 수 있는 노코드(No-code) 인터페이스를 제공합니다. AI 플레이그라운드에서는 유니티 카탈로그 권한 수준에 따른 도구 접근 권한이 자동으로 부여되며, 클로드(Claude)나 GPT 모델과 같은 최첨단 LLM을 이용할 수 있습니다. 에이전트 코드를 본격적으로 작성하기 전에, AI 플레이그라운드를 활용하여 쿼리, LLM, 도구 사용법에 대한 프로토타입을 먼저 제작해보는 것이 좋습니다. 아래는 LLM의 도구 호출을 유도하는 프롬프트를 전송했을 때 AI 플레이그라운드가 어떻게 표시되는지 보여주는 예시입니다.

![ai-playground-tools.png](../Includes/images/ai-playground-tools.png "ai-playground-tools.png")

<p align="center"><em>AI Playground에서는 일부 모델이 GPT-5.1과 같은 도구를 추가할 수 있는 기능을 갖추고 있습니다.</em></p>

## 결론
이제 Databricks에서 UC Tools를 빌드하고, 등록하고, 시각적으로 검사하고, 테스트하는 방법을 이해하셨으니, 이러한 개념들을 실제로 다루는 따라하기 데모를 시작할 준비가 되셨습니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>